# [실습 3] 양자화와 증류를 활용한 경량화 비교

## 실습 개요

> "모델을 작고 빠르게 만들 때 성능 손실은 어떻게 확인해야 할까?"

이번 실습에서는 작은 teacher/student 모델을 만들고, teacher logits를 활용한 증류 손실을 계산합니다. 이어서 동적 양자화를 적용해 모델 파일 크기가 어떻게 달라지는지 비교합니다.

강의에서 다룬 양자화와 증류는 모두 경량화를 위한 기법이지만 접근 방식이 다릅니다. 증류는 작은 모델이 큰 모델의 판단 경향을 배우게 하고, 양자화는 숫자 표현을 줄여 저장 공간과 추론 비용을 낮춥니다.

## 실습 목표

1. teacher와 student 모델의 크기 차이를 확인한다.
2. hard label 손실과 soft target 증류 손실을 구분한다.
3. temperature를 적용한 distillation loss를 구현한다.
4. 동적 양자화 전후 모델 크기를 비교한다.
5. 경량화 결과를 성능과 비용 관점에서 해석한다.

## 데이터 출처

- 데이터: 실습용 벡터 분류 데이터 직접 생성
- 설정 파일: `config/compression_config.json`

---
## 1. 라이브러리와 설정 준비

### 경량화 실험 기준 고정하기

경량화는 모델 크기, 추론 비용, 정확도 사이의 균형을 비교하는 작업입니다. 어떤 temperature와 alpha를 썼는지, student hidden size가 얼마였는지 남기지 않으면 결과를 재현하기 어렵습니다.

설정 파일에는 증류와 양자화 비교에 필요한 기준값을 모아 둡니다. 특히 temperature는 teacher 확률 분포의 부드러움을 바꾸고, alpha는 hard label과 soft target 중 어느 쪽을 더 중시할지 결정하므로 결과 해석에 직접적인 영향을 줍니다.

In [ ]:
import json
import tempfile
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F

config = json.loads(Path('config/compression_config.json').read_text(encoding='utf-8'))
torch.manual_seed(config['seed'])
print('경량화 실습 설정:', config)

---
## 2. teacher와 student 모델 준비

### 크기가 다른 두 모델 비교하기

teacher는 더 큰 hidden size를 가진 모델이고 student는 더 작은 hidden size를 가진 모델입니다. 실제 프로젝트에서는 이미 성능이 검증된 큰 모델을 teacher로 두고, 배포 제약에 맞는 작은 모델을 student로 학습시키는 경우가 많습니다.

여기서는 성능 자체보다 비교의 구조를 보는 것이 목적입니다. 파라미터 수를 먼저 확인하면 student를 따로 두는 이유가 분명해집니다. 작은 모델은 표현력이 제한되지만 저장과 추론 비용에서 이점이 있고, 증류는 그 작은 모델이 teacher의 판단 경향을 최대한 따라가도록 돕습니다.

In [ ]:
class MLPClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_labels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_labels),
        )

    def forward(self, x):
        return self.net(x)


def count_params(model):
    return sum(p.numel() for p in model.parameters())

teacher = MLPClassifier(config['input_dim'], config['teacher_hidden'], config['num_labels'])
student = MLPClassifier(config['input_dim'], config['student_hidden'], config['num_labels'])

print('Teacher 파라미터 수:', count_params(teacher))
print('Student 파라미터 수:', count_params(student))

---
## 3. 실습용 데이터와 teacher 출력 만들기

### soft target이 담고 있는 정보

일반적인 지도학습은 정답 클래스 하나만 hard label로 사용합니다. 반면 증류에서는 teacher의 logits를 soft target으로 사용해 클래스 간 상대적인 선호도까지 student에게 전달합니다.

예를 들어 teacher가 1번 클래스를 가장 높게 보더라도 2번 클래스를 어느 정도 비슷하게 본다면, 그 관계가 확률 분포에 남습니다. student는 이런 분포를 통해 “정답이 무엇인가”뿐 아니라 “헷갈릴 만한 클래스가 무엇인가”도 배울 수 있습니다. 작은 모델일수록 이런 추가 신호가 학습을 부드럽게 만드는 데 도움이 됩니다.

In [ ]:
def make_data(n=128):
    x = torch.randn(n, config['input_dim'])
    rule = torch.randn(config['input_dim'], config['num_labels'])
    y = torch.argmax(x @ rule, dim=-1)
    return x, y

x_train, y_train = make_data(config['num_samples'])

with torch.no_grad():
    teacher_logits = teacher(x_train)

print('입력, 라벨, teacher logits 크기:', x_train.shape, y_train.shape, teacher_logits.shape)

### teacher와 student의 초기 예측 비교하기

증류 학습 전에 두 모델의 예측을 먼저 비교합니다. student는 아직 teacher의 분포를 배운 적이 없으므로 같은 입력에서도 다른 클래스를 고를 수 있습니다.

이 차이가 바로 증류 손실을 넣는 이유입니다. 이후 학습에서는 hard label에 맞추는 손실과 teacher 분포를 따라가는 손실을 함께 사용해, student가 정답과 teacher의 판단 경향을 동시에 반영하도록 만듭니다.

In [ ]:
with torch.no_grad():
    initial_student_logits = student(x_train[:8])
    teacher_predictions = torch.argmax(teacher_logits[:8], dim=-1)
    student_predictions = torch.argmax(initial_student_logits, dim=-1)

print('Teacher 초기 예측:', teacher_predictions.tolist())
print('Student 초기 예측:', student_predictions.tolist())

---
## 4. [TODO] 증류 손실 구현

### [TODO] hard loss와 soft loss를 함께 사용하기

증류 손실은 보통 두 갈래로 구성됩니다. hard loss는 실제 라벨에 대한 cross entropy이고, soft loss는 teacher와 student의 확률 분포를 맞추는 KL divergence입니다. 하나는 정답을 직접 보게 하고, 다른 하나는 teacher가 가진 클래스 간 관계를 전달합니다.

temperature를 적용하면 logits 분포가 부드러워집니다. 값이 커질수록 가장 높은 클래스만 두드러지는 대신 나머지 클래스의 상대적 차이도 더 잘 드러납니다. KL divergence를 계산할 때 student 쪽은 log probability, teacher 쪽은 probability 형태로 넣어야 PyTorch의 `F.kl_div` 사용 방식과 맞습니다.

[TODO]에서는 temperature를 적용한 logits를 만들고, soft loss와 hard loss를 alpha로 섞어 최종 손실을 계산합니다.

완성해야 할 변수는 다음과 같습니다.

- `student_scaled_logits`: `student_logits`를 `temperature`로 나눈 값입니다.
- `teacher_scaled_logits`: `teacher_logits`를 `temperature`로 나눈 값입니다.
- `student_log_probs`: `F.log_softmax(student_scaled_logits, dim=-1)` 결과입니다.
- `teacher_probs`: `F.softmax(teacher_scaled_logits, dim=-1)` 결과입니다.
- `soft_loss`: `F.kl_div(student_log_probs, teacher_probs, reduction='batchmean') * (temperature ** 2)` 결과입니다.
- `total_loss`: `alpha * soft_loss + (1 - alpha) * hard_loss`로 두 손실을 섞은 값입니다.

In [ ]:
def distillation_loss(student_logits, teacher_logits, labels, temperature=2.0, alpha=0.5):
    hard_loss = F.cross_entropy(student_logits, labels)

    # [TODO 1] temperature를 적용한 soft target 손실을 계산하세요.
    # 힌트: student는 log_softmax, teacher는 softmax를 사용하고 KL divergence를 계산합니다.
    student_scaled_logits = None
    teacher_scaled_logits = None
    student_log_probs = None
    teacher_probs = None
    soft_loss = None

    # [TODO 2] hard loss와 soft loss를 alpha 비율로 섞어 최종 손실을 반환하세요.
    total_loss = None
    return total_loss

---
## 5. student 학습과 양자화 비교

### 경량화 후에도 반드시 결과를 다시 확인하기

student를 몇 step 학습한 뒤 동적 양자화를 적용합니다. 동적 양자화는 Linear 레이어의 가중치 표현을 줄이는 방식이라 CPU 추론과 저장 크기 측면에서 효과를 확인하기 좋습니다.

다만 양자화는 항상 이득만 주는 버튼이 아닙니다. 모델 구조, 입력 크기, PyTorch 빌드, 실행 하드웨어에 따라 속도 개선이 작거나 출력 차이가 커질 수 있습니다. 그래서 학습 후 바로 “가벼워졌다”고 판단하지 않고, 저장 크기와 동일 입력에 대한 logits 차이를 함께 확인합니다. 실제 배포에서는 여기에 validation accuracy와 latency 측정까지 붙여야 합니다.

In [ ]:
optimizer = torch.optim.AdamW(student.parameters(), lr=config['learning_rate'])
for epoch in range(config['epochs']):
    optimizer.zero_grad()
    student_logits = student(x_train)
    loss = distillation_loss(
        student_logits,
        teacher_logits.detach(),
        y_train,
        temperature=config['temperature'],
        alpha=config['alpha'],
    )
    loss.backward()
    optimizer.step()
    print(f'증류 학습 epoch={epoch} 손실={loss.item():.4f}')

### 동적 양자화 backend 준비하기

PyTorch의 동적 양자화는 실행 환경에서 지원하는 backend에 영향을 받습니다. 먼저 사용할 수 있는 engine을 확인하고, 가능하면 `qnnpack`을 우선 선택합니다.

환경이 양자화 연산을 지원하지 않는 경우도 실습이 중단되지 않도록 처리했습니다. 그런 상황에서는 원본 student 모델을 그대로 사용하고, 출력 메시지로 양자화를 건너뛰었음을 알려 줍니다. 실무에서도 최적화 코드는 특정 런타임에만 묶이지 않도록 fallback 경로를 두는 편이 안전합니다.

In [ ]:
supported_engines = [engine for engine in torch.backends.quantized.supported_engines if engine != 'none']
if supported_engines:
    preferred_engine = 'qnnpack' if 'qnnpack' in supported_engines else supported_engines[0]
    torch.backends.quantized.engine = preferred_engine
    print('선택된 양자화 engine:', torch.backends.quantized.engine)
else:
    preferred_engine = None
    print('사용 가능한 양자화 engine이 없습니다.')

try:
    quantized_student = torch.quantization.quantize_dynamic(
        student,
        {nn.Linear},
        dtype=torch.qint8,
    )
    quantization_status = '적용됨'
except RuntimeError as error:
    quantized_student = student
    quantization_status = f'건너뜀: {error}'

print('양자화 적용 상태:', quantization_status)

### 저장 크기와 출력 차이 비교하기

양자화의 첫 번째 효과는 저장 크기에서 확인할 수 있습니다. 임시 파일에 원본 모델과 양자화 모델의 state dict를 저장한 뒤 바이트 단위 크기를 비교하면, 배포 패키지가 얼마나 줄었는지 바로 볼 수 있습니다.

하지만 크기가 줄었다고 충분한 검증이 끝난 것은 아닙니다. 같은 입력에 대해 FP32 모델과 양자화 모델의 logits를 함께 출력해 값이 얼마나 달라졌는지 확인해야 합니다. logits 차이가 작더라도 최종 클래스가 바뀔 수 있으므로, 실제 서비스에서는 대표 입력과 검증 데이터셋을 모두 사용해 비교합니다.

In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:
    fp32_path = Path(tmpdir) / 'student_fp32.pt'
    int8_path = Path(tmpdir) / 'student_int8.pt'
    torch.save(student.state_dict(), fp32_path)
    torch.save(quantized_student.state_dict(), int8_path)
    fp32_bytes = fp32_path.stat().st_size
    quantized_bytes = int8_path.stat().st_size
    compression_ratio = quantized_bytes / fp32_bytes
    print('FP32 모델 저장 크기(bytes):', fp32_bytes)
    print('양자화 모델 저장 크기(bytes):', quantized_bytes)
    print('양자화 모델 크기 비율:', round(compression_ratio, 4))

with torch.no_grad():
    fp32_logits = student(x_train[:3])
    quantized_logits = quantized_student(x_train[:3])
print('FP32 모델 logits:', fp32_logits)
print('양자화 모델 logits:', quantized_logits)

### 양자화 출력 차이를 수치로 요약하기

logits를 직접 출력하면 대략적인 차이는 볼 수 있지만, 여러 값이 섞여 있어 한눈에 비교하기 어렵습니다. 평균 절대 차이를 계산하면 FP32 출력과 양자화 출력이 얼마나 벌어졌는지 하나의 숫자로 요약할 수 있습니다.

이 값만으로 품질을 판단할 수는 없습니다. 경량화 검증에서는 출력 차이, 정확도, latency, 메모리 사용량을 함께 봐야 합니다. 그래도 평균 절대 차이는 양자화가 지나치게 큰 변화를 만들었는지 빠르게 확인하는 출발점으로 유용합니다.

In [ ]:
logit_mean_abs_diff = torch.mean(torch.abs(fp32_logits - quantized_logits)).item()
print('FP32와 양자화 logits 평균 절대 차이:', round(logit_mean_abs_diff, 6))